## Local Inference & Evaluation

In [1]:
!pip install transformers sentencepiece -q

import re
import torch
import numpy as np
import pandas as pd

from transformers import (
    AutoTokenizer,
T5ForConditionalGeneration
)

from sklearn.model_selection import train_test_split
import editdistance

In [2]:

DATA_XLSX = "sandhi_data.xlsx"

df = pd.read_excel(DATA_XLSX)

print("Shape:", df.shape)

df = df.rename(columns={
    df.columns[0]: "raw",
    df.columns[1]: "split"
})

print(df.head())
df = df.dropna()
df["raw"] = df["raw"].astype(str)
df["split"] = df["split"].astype(str)
df = df.reset_index(drop=True)
print("After cleaning:", df.shape)

def normalize_split(text):
    text = str(text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

df["split"] = df["split"].apply(
    normalize_split
)
def get_boundary_positions(split_text):
    parts = [
        x.strip()
        for x in split_text.split("+")
    ]
    positions = []
    current = 0
    for i in range(len(parts) - 1):
        current += len(parts[i])
        positions.append(current)
    return positions

word_df = pd.DataFrame({
    "sandhied": df["raw"],
    "gold_split": df["split"]
})

word_df["boundary_positions"] = (
    word_df["gold_split"]
    .apply(get_boundary_positions)
)

print(word_df.head())

Shape: (13930, 2)
                                 raw                               split
0                        प्रथमोऽङ्कः                        प्रथमः+अङ्कः
1                            शब्द इव                            शब्दः+इव
2                             इत इतः                             इतः+इतः
3                            कुतो नु                             कुतः+नु
4  खल्वेष समुत्थितो समुत्थितो ध्वनिः  खलु+एषः+समुत्थितः+समुत्थितः+ध्वनिः
After cleaning: (13925, 2)
                            sandhied                          gold_split  \
0                        प्रथमोऽङ्कः                        प्रथमः+अङ्कः   
1                            शब्द इव                            शब्दः+इव   
2                             इत इतः                             इतः+इतः   
3                            कुतो नु                             कुतः+नु   
4  खल्वेष समुत्थितो समुत्थितो ध्वनिः  खलु+एषः+समुत्थितः+समुत्थितः+ध्वनिः   

  boundary_positions  
0                [6]  
1             

In [3]:
WINDOW_SIZE = 12
def make_window(word, boundary):
    left = max(
        0,
        boundary - WINDOW_SIZE
    )
    right = min(
        len(word),
        boundary + WINDOW_SIZE
    )
    left_context = word[left:boundary]
    right_context = word[boundary:right]
    return (
        left_context
        + " + "
        + right_context
    )

restoration_samples = []

for _, row in word_df.iterrows():
    word = row["sandhied"]
    split = row["gold_split"]
    positions = row["boundary_positions"]
    if len(positions) == 0:
        continue

    boundary = positions[0]
    window = make_window(
        word,
        boundary
    )
    restoration_samples.append({
        "input_text":
            f"split: {window}",
        "target_text":
            split
    })

restoration_df = pd.DataFrame(
    restoration_samples
)
print(restoration_df.head())

train_data, temp_data = train_test_split(
    restoration_samples,
    test_size=0.2,
    random_state=42
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.5,
    random_state=42
)

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))




                  input_text                         target_text
0      split: प्रथमो + ऽङ्कः                        प्रथमः+अङ्कः
1          split: शब्द  + इव                            शब्दः+इव
2           split: इत  + इतः                             इतः+इतः
3          split: कुतो +  नु                             कुतः+नु
4  split: खल् + वेष समुत्थित  खलु+एषः+समुत्थितः+समुत्थितः+ध्वनिः
Train: 11133
Val: 1392
Test: 1392


In [4]:
MODEL_PATH = "./byt5_sandhi_model"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH
)
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_PATH
)
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model.to(device)
model.eval()
print("Model loaded successfully")

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded successfully


In [5]:
def restore_sandhi(window_text):
    input_text = f"split: {window_text}"
    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=64,
            num_beams=4,
            early_stopping=True
        )

    prediction = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
    return prediction

In [6]:
examples = [

    "रामो + हनुमांश्च",

    "प्रथमो + ऽङ्कः",

    "भयं + त्यज",

    "ममा + पि"
]

for ex in examples:

    pred = restore_sandhi(ex)

    print("INPUT :", ex)

    print("OUTPUT:", pred)

    print("-" * 60)

INPUT : रामो + हनुमांश्च
OUTPUT: रामः+अहनुमांश्च
------------------------------------------------------------
INPUT : प्रथमो + ऽङ्कः
OUTPUT: प्रथमः+अङ्कः
------------------------------------------------------------
INPUT : भयं + त्यज
OUTPUT: भयम्+त्यज
------------------------------------------------------------
INPUT : ममा + पि
OUTPUT: मम्+आपि
------------------------------------------------------------


In [7]:
def cer(pred, gold):

    denom = max(
        len(pred),
        len(gold),
        1
    )

    return (
        editdistance.eval(pred, gold)
        / denom
    )

In [8]:
results = []

for sample in test_data:

    inp = sample["input_text"]

    gold = sample["target_text"]

    pred = restore_sandhi(inp.replace("split: ", ""))

    score = cer(pred, gold)

    results.append({

        "input": inp,

        "prediction": pred,

        "gold": gold,

        "cer": score
    })

In [9]:
results_df = pd.DataFrame(results)
results_df.head(10)
avg_cer = results_df["cer"].mean()
print("Average CER:", avg_cer)

Average CER: 0.12469307286539927


In [15]:
results_df.to_csv(

    "byt5_predictions.csv",

    index=False
)

print("Saved predictions")

print("Top 10 best predictions:")
results_df.sort_values(
    "cer"
).head(10)

Saved predictions
Top 10 best predictions:


,input,prediction,gold,cer
494,split: घट + ोपस्थापक,घट+उपस्थापक,घट+उपस्थापक,0.0
871,split: परिणामो + विद्यते,परिणामः+विद्यते,परिणामः+विद्यते,0.0
922,split: तथा + स्य,तथा+अस्य,तथा+अस्य,0.0
923,split: संशयमा + पन्नः,संशयम्+आपन्नः,संशयम्+आपन्नः,0.0
924,split: हेतुर + ुक्तः,हेतुः+उक्तः,हेतुः+उक्तः,0.0
925,split: यज् + ज्ञानं,यत्+ज्ञानं,यत्+ज्ञानं,0.0
912,split: अश्वाघस्य + ात्‌,अश्वाघस्य+आत्,अश्वाघस्य+आत्,0.0
498,split: तादृशो + ऽरसिको,तादृशः+अरसिकः,तादृशः+अरसिकः,0.0
503,split: भिषग् + ‍वरा,भिषक्+वरा,भिषक्+वरा,0.0
489,split: इत् + ‍यधुना,इति+अधुना,इति+अधुना,0.0


In [16]:
print("Top 10 worst predictions:")
results_df.sort_values(
    "cer",
    ascending=False
).head(10)

Top 10 worst predictions:


,input,prediction,gold,cer
1254,split: चतुः +,चतुः+एव,पञ्चाशत्+संख्या,0.866667
589,split: कुण् + ‍ठाऽन्‍त,कुण्+ठाऽन्तः,पुरे+,0.833333
121,split: ्युत्सादयांप + ्रजनयांचिकया,प्रजनयांचिकयाम्,अभ्युत्सादयाम्+प्रजनयाम्+चिकयाम्+रमयाम्+अकः+पा...,0.825000
260,split: पितरो + ह्येषां लुप,पितरः+अह्येषां,पितरः+हि+एषाम्+लुप्तपिण्ड+उदकक्रियाः,0.777778
208,split: कुलक्षयकृतं + दोषं प्रपश्य,कुलक्षयकृतं,कुलक्षयकृतम्+दोषम्+प्रपश्यद्भिः+जन+अर्दन,0.750000
1078,split: यमत्स्यानां + य उपधायाः,यमत्स्यानां,सूर्यतिष्यागस्त्यमत्स्यानाम्+यः+उपधायाः,0.743590
1103,split: ानिर्दिष्टं + समास उपसर्जन,अनिर्दिष्टं,प्रथमानिर्दिष्टम्+समासे+उपसर्जनम्,0.727273
155,split: व्यधो + मन्थो मन्थो,व्यधः+अन्थः,व्यधः+मन्थः+मन्थः+ग्रहः+ग्रहः+दाहः+च,0.722222
222,split: प्रयोजनं + वाच्यं वाच्य,प्रयोजनं,प्रयोजनम्+वाच्यम्+वाच्यम्,0.720000
276,split: दोषावस्कराद् + वुन्,दोषावस्करात्+एव,पूर्वाह्णापराह्णार्द्रामूलप्रदोषावस्करात्+वुन्,0.717391
